# Set Up pwd and auto updates

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


# Ensure Data Exists 

In [ ]:
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry


# Exparimentation with season functionality

In [ ]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

import seaborn as sns

# Creating Schedules for TM 

In [37]:
traffic_percentile_schedule = ScheduleSpecs(
     mode='static',
    value=90,  
    dist=None
)

bus_interval_schedule = ScheduleSpecs(
    mode='static',
    value=10,  # static bus interval of 15 minutes
    dist=None
)

crashes_schedule = ScheduleSpecs(
    mode='static',
    value=0,  # static bus interval of 15 minutes
    dist=None # normal distribution for crashes per 100k VMT
)



# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [38]:
pop_params = PopulationParams(
    population_size=1500,
    prior_car=22.0,
    prior_bus=27.0,
    time_decay_rate=0.1,
    prior_weight=1.0,
    uncertainty_multiplier=1.0,
)


config = make_season_config(
    season_id='test2k',
    run_description='',
    seed=33,
    n_days=2,
    max_steps=20000,
    max_persons=99999,
    collect_every_n=60,
    start_hr=8,
    bus_capacity=60,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='volume',
    toll_params={"volume_threshold": 100, "slope": 0.05,"base_price": 5.0},   #{'car': 5.0, 'bus': 0.0}, {"volume_threshold": 100, "slope": 0.05,"base_price": 5.0}
    canyon_closures_schedule=None,
    traffic_percentile_schedule=traffic_percentile_schedule,
    bus_interval_schedule=bus_interval_schedule,
    crashes_schedule=crashes_schedule, 
    population_params=pop_params
   
)

In [39]:
import cProfile
import pstats

def main():
    # Example usage of SeasonOrchestrator with example_config
    orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
    orchestrator.run_season()

if __name__ == "__main__":
    prof = cProfile.Profile()
    prof.enable()

    main()

    prof.disable()
    prof.dump_stats("prof.stats")

    p = pstats.Stats("prof.stats")
    p.strip_dirs().sort_stats("cumulative").print_stats(40)




Season outputs will be saved to: data/season_outputs/test2k


Simulating:  44%|████▍     | 8763/20000 [02:30<03:13, 58.09step/s]


1500 people arrived stopping model.
Day:0: N:1500, Avg TT:29.6 min, Avg_cumtime_lost:2.2 min, Avg Cost (VOT standardized):$23.9, Avg Realized Cost:$27.5, bus_share:0.45, avg_tt_bus:40.2 min, avg_tt_car:21.0 min, avg_toll_car:$7.64, Total toll:$6351.25, 



Simulating:  36%|███▌      | 7237/20000 [02:11<03:51, 55.13step/s]


1500 people arrived stopping model.
Day:1: N:1500, Avg TT:25.7 min, Avg_cumtime_lost:3.1 min, Avg Cost (VOT standardized):$23.1, Avg Realized Cost:$27.3, bus_share:0.36, avg_tt_bus:32.0 min, avg_tt_car:22.1 min, avg_toll_car:$9.29, Total toll:$8931.40, 

Season Summary - Days Run: 2, Total Trips: 3000, 
Avg TT (all): 27.60, 
--- Cost Metrics --- 
     Marginal, Std VOT: $10.16, 
     Std VOT: $23.50, 
     All, Agent VOT: $27.41, 
     Bus, Agent VOT: $21.66, 
     Car, Agent VOT: $31.29, 
--- Tolling Metrics --- 
Avg Toll Cars: $8.53
Total Revenue: $15282.65
Fri Dec 19 12:03:27 2025    prof.stats

         1862630769 function calls (1862317926 primitive calls) in 282.790 seconds

   Ordered by: cumulative time
   List reduced from 1614 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000  282.816  282.816 4179568427.py:4(main)
        1    0.000    0.000  282.345  282.345 season_orchestrator.py:57(run_season

In [ ]:
# todo evaluate  